# SafeCrop · Step 2b: Gemini gets the same reference photos as the CLIP database (Google Colab)

This notebook repeats the Step 2 test with **one new predictor**. CLIP zero-shot and the CLIP photo database work exactly as in Step 2.

| Predictor | What it gets | Choices |
|---|---|---|
| **CLIP zero-shot** | the photo; compares it with short texts such as "a photo of jute mosaic" | 33 |
| **CLIP photo database** | the photo; the 5 most similar **clean** photos of *other* plants vote (a photo's own clean version is never used) | the 7 diseases in the dataset |
| **Gemini 3.5 Flash-Lite** | the photo and the 33 options, names only (answers reused from Step 2) | 33 |
| **Gemini + references + descriptions** (new) | the photo, **the same 5 reference photos the CLIP database votes with**, each with its label, and the 33 options, each with **a one-line description** of what it looks like | 33 |

**Why:** the CLIP database is given labelled example photos, while plain Gemini is given none. Here Gemini gets the same examples, so the comparison is about who uses the same information better. The descriptions help Gemini with the diseases it knows little about (the info-scarce group).

**Be fair when you report it:** the reference photos only come from our 7 diseases, so they hint at which answers are likely. The CLIP database has that same hint. The notebook also shows how often Gemini simply agreed with the database, and how it did when it disagreed.

### Before you run (about 5 minutes)

1. **GPU (optional, makes CLIP faster):** *Runtime → Change runtime type → T4 GPU → Save*. CPU also works.
2. **Gemini API key:** the same Colab secret as Step 2: `GEMINI_API_KEY` in the **key icon (Secrets)** on the left, with **Notebook access** on. A new key comes from [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
3. **Your images:** `noisy_images.zip` anywhere in your Google Drive, as in Step 2.
4. Click **Runtime → Run all**.

**How long it takes:** the photo and its 5 reference photos go to Gemini in **one** request, so the new predictor needs one request per image. All 840 images need 840 requests, which is about two days on the free key (500 a day). To test severe noise only, set `GEMINI_LEVELS = ["L3"]` in Settings: 210 requests, done in one day. The plain Gemini answers are reused from Step 2 (`SafeCrop/results/gemini_3.5_flash_lite_predictions.csv` in your Drive), so they cost nothing. If the daily limit is reached, the notebook saves everything and still makes the tables. Running it again continues where it stopped.

**Where results go:** `SafeCrop/results/gemini_help` in your Drive, so the Step 2 tables are not overwritten.

In [ ]:
# Install anything missing (Colab already has most of it). Safe to run again.
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def ensure(pkg, min_major=None):
    try:
        v = version(pkg)
        if min_major is not None and int(v.split(".")[0]) < min_major:
            raise PackageNotFoundError(pkg)
    except PackageNotFoundError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=True)

ensure("google-genai", min_major=1)
ensure("transformers")
ensure("openpyxl")
print("google-genai", version("google-genai"), "| transformers", version("transformers"))

In [ ]:
import os, re, csv, json, time, glob, random, shutil, zipfile, datetime
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image as ShowImage
from tqdm.auto import tqdm

def short(e, n=300):
    s = str(e).replace("\n", " ")
    return s if len(s) <= n else s[:n] + " ..."

print("pandas", pd.__version__, "| numpy", np.__version__)

## Settings

You can leave these as they are.

In [ ]:
# ---- where the images are ----
SOURCE = "auto"      # "auto": look in Google Drive, ask for an upload if not found | "drive" | "upload"
DATASET_PATH = ""    # optional exact path, e.g. "/content/drive/MyDrive/safecrop dataset/Noisy images/noisy_images.zip"

# ---- CLIP (the same as Step 2) ----
RUN_CLIP = True
CLIP_MODEL = "openai/clip-vit-large-patch14"
CLIP_DATABASE_NEIGHBOURS = 5   # how many similar clean photos vote in the database method
CLIP_REDO = False   # False = reuse this notebook's saved CLIP results if they are complete

# ---- Gemini ----
RUN_GEMINI = True
GEMINI_MODEL = "gemini-3.5-flash-lite"   # free key: 15 requests a minute, 500 a day
GEMINI_REQUESTS_PER_MINUTE = 14          # just under the free limit of 15
GEMINI_PHOTOS_PER_DISEASE = None         # None = every photo; e.g. 10 = 10 photos per disease, to save quota
GEMINI_LEVELS = ["L0", "L1", "L2", "L3"] # e.g. ["L3"] = severe noise only (210 requests instead of 840)

# Plain Gemini (Step 2): names only, no reference photos
GEMINI_PLAIN = "reuse"   # "reuse": use the answers Step 2 saved (no new requests) | "run": also ask for missing images | "off"

# Gemini with help (new in this notebook)
GEMINI_HELP = "both"     # "both": reference photos + descriptions | "references" | "descriptions" | "off"
REFERENCE_PHOTOS = CLIP_DATABASE_NEIGHBOURS   # the same photos the CLIP database votes with
REFERENCE_MAX_SIDE = 768                      # reference photos are shrunk to this size (pixels) to keep requests small

SEED = 299

# ---- check the settings ----
if GEMINI_PLAIN not in ("reuse", "run", "off"):
    raise ValueError('GEMINI_PLAIN must be "reuse", "run" or "off"')
if GEMINI_HELP not in ("both", "references", "descriptions", "off"):
    raise ValueError('GEMINI_HELP must be "both", "references", "descriptions" or "off"')
if not GEMINI_LEVELS or set(GEMINI_LEVELS) - {"L0", "L1", "L2", "L3"}:
    raise ValueError('GEMINI_LEVELS must list some of "L0", "L1", "L2", "L3"')
if REFERENCE_PHOTOS > CLIP_DATABASE_NEIGHBOURS:
    raise ValueError("REFERENCE_PHOTOS can't be more than CLIP_DATABASE_NEIGHBOURS (Gemini gets the database's own photos)")
HELP_REFS = REFERENCE_PHOTOS if GEMINI_HELP in ("both", "references") else 0
HELP_DESC = GEMINI_HELP in ("both", "descriptions")
print("Plain Gemini:", GEMINI_PLAIN, "| Gemini with help:", GEMINI_HELP,
      f"({HELP_REFS} reference photos, descriptions {'on' if HELP_DESC else 'off'}) | levels:", ", ".join(GEMINI_LEVELS))

## 1. Get the images

This looks for `noisy_images.zip`, or an unzipped folder with `metadata.csv`, in your Google Drive. If it finds several, it uses the newest. If Drive isn't available, a **Choose files** button appears: pick `noisy_images.zip`. Also pick `gemini_3.5_flash_lite_predictions.csv` (Step 2's plain Gemini answers) and, when continuing, this notebook's `..._help_..._predictions.csv`, if you work without Drive.

Results are saved to `SafeCrop/results/gemini_help` in your Drive (Step 2's results in `SafeCrop/results` are only read, never overwritten), or to Colab's temporary storage if Drive isn't connected. They are downloaded as a zip at the end.

In [ ]:
WORK = "/content/safecrop_inference"
DATA_DIR = os.path.join(WORK, "data")
os.makedirs(DATA_DIR, exist_ok=True)

def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Couldn't connect to Google Drive:", short(e, 200))
    return os.path.isdir("/content/drive/MyDrive")

def walk(top, max_depth):
    # os.walk that stops max_depth folders below `top` and skips hidden folders
    top = top.rstrip(os.sep)
    base = top.count(os.sep)
    for root, dirs, files in os.walk(top):
        depth = root.count(os.sep) - base
        dirs[:] = sorted(d for d in dirs if not d.startswith(".")) if depth < max_depth else []
        yield root, files

def is_dataset_folder(root, files):
    return "metadata.csv" in files and os.path.isdir(os.path.join(root, "L0"))

def find_dataset_folder(top):
    for root, files in walk(top, max_depth=6):
        if is_dataset_folder(root, files):
            return root
    return None

def unpack(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist()
                   if not m.startswith("__MACOSX") and not os.path.basename(m.rstrip("/")).startswith("._")]
        z.extractall(dest, members)

def prepare(path):
    # returns the folder that holds metadata.csv and L0..L3, unzipping first if needed
    if os.path.isdir(path):
        return find_dataset_folder(path)
    if not (os.path.isfile(path) and zipfile.is_zipfile(path)):
        return None
    dest = os.path.join(DATA_DIR, re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(path))[0]))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    print("Unzipping", os.path.basename(path), "...")
    unpack(path, dest)
    for _ in range(3):   # handles a zip inside the zip, e.g. the whole 'safecrop dataset' folder downloaded from Drive
        found = find_dataset_folder(dest)
        if found:
            return found
        inner = [os.path.join(r, f) for r, fs in walk(dest, 6) for f in fs if f.lower().endswith(".zip")]
        inner = [p for p in inner if "noisy" in os.path.basename(p).lower()] or inner
        if not inner:
            break
        for p in inner:
            unpack(p, os.path.splitext(p)[0])
            os.remove(p)
    return None

def find_in_drive():
    found = []
    for root, files in walk("/content/drive/MyDrive", max_depth=4):
        if is_dataset_folder(root, files):
            found.append(root)
        for f in files:
            fl = f.lower()
            if fl.endswith(".zip") and "noisy" in fl and "result" not in fl:
                found.append(os.path.join(root, f))
    return sorted(found, key=os.path.getmtime, reverse=True)

DRIVE_OK, dataset_dir, uploaded = False, None, []

if DATASET_PATH:
    if DATASET_PATH.startswith("/content/drive"):
        DRIVE_OK = mount_drive()
    dataset_dir = prepare(DATASET_PATH)
    if dataset_dir is None:
        print("No images found at DATASET_PATH:", DATASET_PATH)

if dataset_dir is None and SOURCE in ("auto", "drive"):
    DRIVE_OK = mount_drive()
    if DRIVE_OK:
        print("Searching your Google Drive for noisy_images ...")
        candidates = find_in_drive()
        for c in candidates:
            print("  found:", c)
        for c in candidates:
            dataset_dir = prepare(c)
            if dataset_dir:
                print("Using:", c)
                break
        if dataset_dir is None:
            print("No noisy_images.zip found in your Drive. (Files in 'Shared with me' only count if you add a shortcut to My Drive.)")

if dataset_dir is None and SOURCE in ("auto", "upload"):
    from google.colab import files
    print("Click 'Choose files' and pick noisy_images.zip "
          "(add gemini_3.5_flash_lite_predictions.csv from Step 2, and this notebook's help predictions "
          "if you are continuing an earlier run without Drive).")
    os.makedirs(os.path.join(WORK, "uploads"), exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join(WORK, "uploads", name)
        with open(p, "wb") as f:
            f.write(data)
        uploaded.append(p)
    for p in uploaded:
        if p.lower().endswith(".zip"):
            dataset_dir = prepare(p)
            if dataset_dir:
                break

if dataset_dir is None:
    raise RuntimeError("No images found. Put noisy_images.zip in your Google Drive (or set SOURCE = 'upload') and run again.")

# ---- read the list of images ----
meta = pd.read_csv(os.path.join(dataset_dir, "metadata.csv"))
need = {"image", "photo_id", "disease", "level"}
if need - set(meta.columns):
    raise RuntimeError(f"metadata.csv is missing the columns {sorted(need - set(meta.columns))}")
for col in ["disease_name", "group", "origin", "noise"]:
    if col not in meta.columns:
        meta[col] = ""
meta = meta.fillna("")
meta["disease_name"] = [n if n else d for n, d in zip(meta.disease_name, meta.disease)]
meta["level"] = meta.level.astype(str)
meta["path"] = [os.path.join(dataset_dir, p) for p in meta.image]
missing = ~meta.path.map(os.path.exists)
if missing.any():
    print(f"Warning: {missing.sum()} images listed in metadata.csv are missing and will be skipped.")
    meta = meta[~missing]
meta = meta.drop_duplicates("image").sort_values(["disease", "photo_id", "level"]).reset_index(drop=True)

# ---- where results go ----
RESULTS_DIR = "/content/drive/MyDrive/SafeCrop/results" if DRIVE_OK else os.path.join(WORK, "results")
OUT_DIR = os.path.join(RESULTS_DIR, "gemini_help")       # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)
GEMINI_CACHE = os.path.join(RESULTS_DIR, "gemini_3.5_flash_lite_predictions.csv")   # plain Gemini, shared with Step 2
HELP_CACHE = os.path.join(OUT_DIR, f"gemini_3.5_flash_lite_help_{GEMINI_HELP}_predictions.csv")
CLIP_FILE = os.path.join(OUT_DIR, "clip_predictions.csv")
NEIGHBOURS_FILE = os.path.join(OUT_DIR, "clip_database_neighbours.csv")   # the reference photos of every image

def merge_upload(src, dest):
    old = pd.read_csv(src, dtype=str, keep_default_na=False)
    if os.path.exists(dest):
        old = pd.concat([pd.read_csv(dest, dtype=str, keep_default_na=False), old])
    old.drop_duplicates(["image", "model"], keep="last").to_csv(dest, index=False)
    print("Continuing from", os.path.basename(src))

for p in uploaded:   # uploaded answers from earlier runs are merged in, so nothing is asked twice
    name = os.path.basename(p).lower()
    if name.endswith(".csv") and "gemini" in name:
        merge_upload(p, HELP_CACHE if "help" in name else GEMINI_CACHE)

print(f"\n{len(meta)} images, {meta.photo_id.nunique()} photos, {meta.disease.nunique()} diseases. Results go to {OUT_DIR}")
display(meta.pivot_table(index=["group", "disease_name"], columns="level", values="image", aggfunc="count", fill_value=0))

## 2. The 33 answer options and their descriptions

These are the 8 SafeCrop diseases (marked ✓ when they're in your dataset), common look-alike diseases of the same crops, and "healthy" for each crop. Every predictor except the CLIP database chooses from this list.

CLIP zero-shot and plain Gemini get **only the names**, as in Step 2. The new Gemini predictor also gets the **one-line description** next to each option, saying what it looks like on the plant. Descriptions are written for how symptoms look in a photo: colour, shape, and where on the plant. They don't tell the model anything about our photos.

In [ ]:
# key, crop, name shown to Gemini (after "Crop: "), full name used in CLIP's sentences
LABELS = [dict(key=k, crop=c, gemini=g, clip=n) for k, c, g, n in [
    ("apple_scab",                      "apple",      "scab",                                  "apple scab"),
    ("apple_cedar_rust",                "apple",      "cedar apple rust",                      "cedar apple rust"),
    ("apple_alternaria_leaf_spot",      "apple",      "Alternaria leaf spot",                  "apple Alternaria leaf spot"),
    ("apple_powdery_mildew",            "apple",      "powdery mildew",                        "apple powdery mildew"),
    ("apple_healthy",                   "apple",      "healthy",                               "healthy apple"),
    ("potato_late_blight",              "potato",     "late blight",                           "potato late blight"),
    ("potato_early_blight",             "potato",     "early blight",                          "potato early blight"),
    ("potato_healthy",                  "potato",     "healthy",                               "healthy potato"),
    ("grape_leaf_blight",               "grape",      "leaf blight (Isariopsis leaf spot)",    "grape leaf blight"),
    ("grape_black_rot",                 "grape",      "black rot",                             "grape black rot"),
    ("grape_esca",                      "grape",      "esca (black measles)",                  "grape esca"),
    ("grape_downy_mildew",              "grape",      "downy mildew",                          "grape downy mildew"),
    ("grape_healthy",                   "grape",      "healthy",                               "healthy grape"),
    ("pear_leaf_spot",                  "pear",       "leaf spot",                             "pear leaf spot"),
    ("pear_scab",                       "pear",       "scab",                                  "pear scab"),
    ("pear_fire_blight",                "pear",       "fire blight",                           "pear fire blight"),
    ("pear_rust",                       "pear",       "rust",                                  "pear rust"),
    ("pear_healthy",                    "pear",       "healthy",                               "healthy pear"),
    ("rice_stem_borer",                 "rice",       "stem borer damage (dead heart / white head)", "rice stem borer damage"),
    ("rice_blast",                      "rice",       "blast",                                 "rice blast"),
    ("rice_brown_spot",                 "rice",       "brown spot",                            "rice brown spot"),
    ("rice_bacterial_leaf_blight",      "rice",       "bacterial leaf blight",                 "rice bacterial leaf blight"),
    ("rice_sheath_blight",              "rice",       "sheath blight",                         "rice sheath blight"),
    ("rice_healthy",                    "rice",       "healthy",                               "healthy rice"),
    ("jute_mosaic",                     "jute",       "mosaic",                                "jute mosaic"),
    ("jute_dieback",                    "jute",       "dieback",                               "jute dieback"),
    ("jute_stem_rot",                   "jute",       "stem rot",                              "jute stem rot"),
    ("jute_anthracnose",                "jute",       "anthracnose",                           "jute anthracnose"),
    ("jute_healthy",                    "jute",       "healthy",                               "healthy jute"),
    ("black_gram_yellow_mosaic",        "black gram", "yellow mosaic",                         "black gram yellow mosaic"),
    ("black_gram_cercospora_leaf_spot", "black gram", "Cercospora leaf spot",                  "black gram Cercospora leaf spot"),
    ("black_gram_powdery_mildew",       "black gram", "powdery mildew",                        "black gram powdery mildew"),
    ("black_gram_healthy",              "black gram", "healthy",                               "healthy black gram"),
]]

# one line per option: what it looks like on the plant (only the new Gemini predictor sees these)
DESCRIPTIONS = {
    "apple_scab": "olive-green to dark brown velvety spots on the leaves that turn dark and scabby; leaves may pucker",
    "apple_cedar_rust": "bright yellow-orange round spots on the upper leaf, often with a red rim; small tubes or cups underneath",
    "apple_alternaria_leaf_spot": "many small round brown spots with a darker purple-brown rim that can merge into larger dead patches",
    "apple_powdery_mildew": "white powdery coating on leaves and shoot tips; leaves narrow, curled and crinkled",
    "apple_healthy": "evenly green apple leaves with no spots, coating or discoloured patches",
    "potato_late_blight": "large dark brown to black water-soaked patches, often from the leaf tip or edge, with a pale border and white mould underneath in damp weather",
    "potato_early_blight": "brown spots with rings inside like a target, mostly on older lower leaves, often with yellow around them",
    "potato_healthy": "evenly green potato leaves with no spots, blotches or yellowing",
    "grape_leaf_blight": "irregular dark brown to black angular spots that join into large dry dead areas on the leaf",
    "grape_black_rot": "round tan to brown spots with a dark brown rim and tiny black dots inside",
    "grape_esca": "yellow or red-brown stripes between the veins with green along the veins (tiger stripes), parts drying out",
    "grape_downy_mildew": "yellow oily-looking patches on the upper leaf with white downy growth underneath",
    "grape_healthy": "evenly green grape leaves with no spots, stripes or mould",
    "pear_leaf_spot": "many small round grey to brown spots with a dark rim scattered over the leaf",
    "pear_scab": "olive-black velvety spots on leaves and fruit, like apple scab",
    "pear_fire_blight": "leaves and shoot tips turn black or brown as if burnt and stay attached; shoot tip bends like a hook",
    "pear_rust": "bright orange spots on the upper leaf that later form raised bumps or horn-like growths underneath",
    "pear_healthy": "evenly green pear leaves with no spots, scorching or rust",
    "rice_stem_borer": "pest damage: the central shoot dries and turns brown while outer leaves stay green (dead heart), or the grain head stays upright, empty and white (white head)",
    "rice_blast": "diamond or spindle-shaped spots with grey or white centres and brown edges on the leaves",
    "rice_brown_spot": "many small oval brown spots, often with a grey centre and a yellow halo, scattered over the leaves",
    "rice_bacterial_leaf_blight": "yellow to white wavy stripes from the leaf tip and edges; leaves dry out from the tip down",
    "rice_sheath_blight": "oval grey-green to white patches with brown edges on the leaf sheath near the water, like snake skin",
    "rice_healthy": "green upright rice plants with no spots, stripes, dried shoots or white heads",
    "jute_mosaic": "leaves with a patchy yellow and green mosaic pattern, sometimes crinkled or smaller than normal",
    "jute_dieback": "shoots and branches dry out and die from the tip downward; leaves wilt and turn brown or black",
    "jute_stem_rot": "dark brown to black patches on the stem that spread, so the plant wilts or breaks; leaves may get brown spots",
    "jute_anthracnose": "sunken dark brown spots or cracks on the stem and brown spots on the leaves",
    "jute_healthy": "evenly green jute leaves and stems with no spots, mosaic or drying",
    "black_gram_yellow_mosaic": "bright yellow patches mixed with green across the leaves, spreading until leaves turn mostly yellow",
    "black_gram_cercospora_leaf_spot": "small round brown spots with grey centres and red-brown rims on the leaves",
    "black_gram_powdery_mildew": "white powdery patches on the leaves",
    "black_gram_healthy": "evenly green black gram leaves with no spots, yellowing or powder",
}

# a disease folder that isn't in the list above still works: it gets its own option
known = {l["key"] for l in LABELS}
for d, name in meta[["disease", "disease_name"]].drop_duplicates().itertuples(index=False):
    if d not in known:
        crop = name.split()[0].lower() if name else "plant"
        LABELS.append(dict(key=d, crop=crop, gemini=name, clip=name.lower()))
        DESCRIPTIONS.setdefault(d, "")
        print("Added an option for", d)

LABEL_KEYS = [l["key"] for l in LABELS]
LABEL_CROP = {l["key"]: l["crop"] for l in LABELS}
OPTIONS = [f"{l['crop'][0].upper() + l['crop'][1:]}: {l['gemini']}" for l in LABELS]   # what Gemini sees
OPTION_TO_KEY = dict(zip(OPTIONS, LABEL_KEYS))
KEY_TO_OPTION = dict(zip(LABEL_KEYS, OPTIONS))
TARGETS = sorted(meta.disease.unique())

options_table = pd.DataFrame({"option": OPTIONS, "in your dataset": ["✓" if k in TARGETS else "" for k in LABEL_KEYS],
                              "description (new Gemini predictor only)": [DESCRIPTIONS.get(k, "") for k in LABEL_KEYS]})
missing_desc = [k for k in LABEL_KEYS if not DESCRIPTIONS.get(k)]
if missing_desc:
    print("No description yet for:", ", ".join(missing_desc), "(these options are shown by name only)")
print(len(LABELS), "options;", len(TARGETS), "of them are diseases in your dataset")
with pd.option_context("display.max_colwidth", 140):
    display(options_table)

## 3. CLIP

**Zero-shot:** each option is written as four short sentences, such as *"a photo of jute mosaic."* and *"a close-up photo of a leaf with jute mosaic."*. CLIP turns these sentences and every photo into vectors and picks the option whose sentences are closest to the photo.

**Photo database:** CLIP compares each image with the **clean** (L0) photos of every *other* plant in the dataset. The 5 most similar ones vote, weighted by similarity. The photo's own clean version is always left out, otherwise the noisy copy would simply find its original. This is the matching idea of SafeCrop's planned CLIP model, before any fine-tuning.

**New here:** for every image, the notebook also saves **which 5 clean photos voted** (`clip_database_neighbours.csv`). These are exactly the reference photos the new Gemini predictor gets, so both see the same examples. The CLIP predictions themselves are computed the same way as in Step 2.

In [ ]:
CLIP_OK = False
need_clip = RUN_CLIP
def clip_saved():
    # True when this notebook already saved complete CLIP results and reference photos for these images
    if not (os.path.exists(CLIP_FILE) and os.path.exists(NEIGHBOURS_FILE)):
        return False
    old = pd.read_csv(CLIP_FILE, keep_default_na=False)
    nb = pd.read_csv(NEIGHBOURS_FILE, keep_default_na=False)
    return (set(meta.image) <= set(old.image) and "clip_model" in old and (old.clip_model == CLIP_MODEL).all()
            and set(meta.image) <= set(nb.image) and (nb.clip_model == CLIP_MODEL).all()
            and nb.groupby("image").size().min() >= CLIP_DATABASE_NEIGHBOURS)

if RUN_CLIP and not CLIP_REDO and clip_saved():
    print("CLIP results and reference photos for these images are already saved, so CLIP is skipped this time "
          "(set CLIP_REDO = True in Settings to run it again).")
    need_clip = False
if need_clip:
    try:
        import torch
        from transformers import CLIPModel, AutoTokenizer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading {CLIP_MODEL} on " + ("the GPU" if device == "cuda" else
              "the CPU (slower; Runtime > Change runtime type > T4 GPU makes it faster)") + " ...")
        clip_model = CLIPModel.from_pretrained(CLIP_MODEL)
        clip_model = (clip_model.half() if device == "cuda" else clip_model.float()).to(device).eval()
        clip_tok = AutoTokenizer.from_pretrained(CLIP_MODEL)
        CLIP_OK = True
    except Exception as e:
        print("Couldn't load CLIP:", short(e))
        print("CLIP is skipped this time; Gemini still runs. Try again later with Runtime > Run all.")

if CLIP_OK:
    size = clip_model.config.vision_config.image_size
    MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
    STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)
    dtype = next(clip_model.parameters()).dtype

    def load_for_clip(path):
        img = Image.open(path).convert("RGB")
        w, h = img.size
        s = size / min(w, h)
        if s != 1:
            img = img.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BICUBIC)
        w, h = img.size
        left, top = (w - size) // 2, (h - size) // 2
        img = img.crop((left, top, left + size, top + size))
        x = torch.from_numpy(np.asarray(img).astype(np.float32) / 255.0).permute(2, 0, 1)
        return (x - MEAN) / STD

    @torch.no_grad()
    def image_vectors(paths, batch=32):
        out = []
        for i in tqdm(range(0, len(paths), batch), desc="CLIP: images"):
            x = torch.stack([load_for_clip(p) for p in paths[i:i + batch]]).to(device=device, dtype=dtype)
            v = clip_model.visual_projection(clip_model.vision_model(pixel_values=x).pooler_output)
            out.append(torch.nn.functional.normalize(v.float(), dim=-1).cpu())
        return torch.cat(out).numpy()

    @torch.no_grad()
    def text_vectors(texts):
        t = clip_tok(texts, padding=True, truncation=True, max_length=77, return_tensors="pt")
        v = clip_model.text_projection(clip_model.text_model(input_ids=t["input_ids"].to(device),
                                                            attention_mask=t["attention_mask"].to(device)).pooler_output)
        return torch.nn.functional.normalize(v.float(), dim=-1).cpu().numpy()

    def sentences(l):
        if l["key"].endswith("_healthy"):
            c = l["crop"]
            return [f"a photo of a healthy {c} plant.", f"a photo of healthy {c} leaves.",
                    f"a close-up photo of a healthy {c} leaf.", f"a photo of a {c} plant with no disease."]
        n, c = l["clip"], l["crop"]
        return [f"a photo of {n}.", f"a photo of a plant with {n}.",
                f"a close-up photo of a leaf with {n}.", f"a photo of {c} plants affected by {n}."]

    # one vector per option: the average of its four sentences
    T = np.stack([text_vectors(sentences(l)).mean(axis=0) for l in LABELS])
    T /= np.linalg.norm(T, axis=1, keepdims=True)
    V = image_vectors(list(meta.path))

    # ---- zero-shot: closest option ----
    scale = float(clip_model.logit_scale.exp().item())
    logits = scale * V @ T.T
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs /= probs.sum(axis=1, keepdims=True)
    order = np.argsort(-probs, axis=1)
    zs = pd.DataFrame({
        "image": meta.image, "model": "CLIP zero-shot",
        "pred_label": [LABEL_KEYS[i] for i in order[:, 0]],
        "confidence": probs[np.arange(len(meta)), order[:, 0]],
        "top3": ["; ".join(LABEL_KEYS[i] for i in row[:3]) for row in order],
    })

    # ---- photo database: the most similar clean photos of OTHER plants vote ----
    ref = np.flatnonzero((meta.level == "L0").to_numpy())
    ref_photo = meta.photo_id.to_numpy()[ref]
    ref_disease = meta.disease.to_numpy()[ref]
    S = V @ V[ref].T
    S[meta.photo_id.to_numpy()[:, None] == ref_photo[None, :]] = -np.inf     # never the photo's own clean version
    k = max(1, min(CLIP_DATABASE_NEIGHBOURS, len(ref) - 1))
    nearest = np.argsort(-S, axis=1)[:, :k]
    db_pred, db_conf, db_top = [], [], []
    for i in range(len(meta)):
        votes = {}
        for j in nearest[i]:
            votes[ref_disease[j]] = votes.get(ref_disease[j], 0.0) + max(float(S[i, j]), 0.0) + 1e-6
        ranked = sorted(votes.items(), key=lambda kv: -kv[1])
        db_pred.append(ranked[0][0])
        db_conf.append(ranked[0][1] / sum(votes.values()))
        db_top.append("; ".join(d for d, _ in ranked[:3]))
    db = pd.DataFrame({"image": meta.image, "model": "CLIP photo database",
                       "pred_label": db_pred, "confidence": db_conf, "top3": db_top})

    # ---- save the photos that voted: the new Gemini predictor gets exactly these as reference photos ----
    ref_image = meta.image.to_numpy()[ref]
    nb_rows = [(meta.image.iat[i], r + 1, ref_image[j], ref_disease[j], round(float(S[i, j]), 4))
               for i in range(len(meta)) for r, j in enumerate(nearest[i])]
    neighbours = pd.DataFrame(nb_rows, columns=["image", "rank", "ref_image", "ref_disease", "similarity"])
    neighbours["clip_model"] = CLIP_MODEL
    neighbours.to_csv(NEIGHBOURS_FILE, index=False)
    assert not (neighbours.ref_image.map(dict(zip(meta.image, meta.photo_id)))
                == neighbours.image.map(dict(zip(meta.image, meta.photo_id)))).any(), "a photo was its own reference"

    clip_preds = pd.concat([zs, db], ignore_index=True)
    clip_preds["clip_model"] = CLIP_MODEL
    clip_preds.to_csv(CLIP_FILE, index=False)

    quick = clip_preds.merge(meta[["image", "disease", "level"]], on="image")
    quick["correct"] = quick.pred_label == quick.disease
    print("\nCLIP accuracy (%) by level:")
    display((quick.groupby(["model", "level"]).correct.mean() * 100).round(1).unstack("level"))
    print("Saved", CLIP_FILE, "and", NEIGHBOURS_FILE)

    del clip_model
    if device == "cuda":
        torch.cuda.empty_cache()

## 4. Gemini 3.5 Flash-Lite: plain, and with reference photos and descriptions

**Plain Gemini** is the Step 2 test: the photo and the 33 option names. Its saved answers are reused, so it costs no quota (unless `GEMINI_PLAIN = "run"`).

**Gemini with help** gets one request per image with:

1. the farmer's photo (the noisy image, exactly as plain Gemini sees it);
2. the **5 reference photos the CLIP database voted with** for that image, most similar first, each with its label, for example *"Reference photo 1 (most similar). Label: Apple: scab"*. They are clean photos of *other* plants; the image's own clean photo is never one of them;
3. the 33 options, each with its one-line description.

The answer comes back as the same JSON as before (diagnosis, confidence, clear enough or not), so both Gemini predictors are scored the same way.

* **Speed and limits:** 14 requests a minute; if the free daily quota runs out, the notebook stops politely and continues on the next run (the quota resets at midnight Pacific time, about 2 pm in Bangladesh).
* Images go in a fixed random order, mixing photos and levels, so even a partial run gives balanced tables.
* The reference photos are shrunk to at most 768 pixels to keep requests small; the farmer's photo is sent unchanged.
* Every answer is saved with the reference photos it was given (`references` column), so you can check any answer later.

In [ ]:
import io

PLAIN_NAME = "Gemini 3.5 Flash-Lite"
HELP_NAME = {"both": "Gemini + references + descriptions", "references": "Gemini + references",
             "descriptions": "Gemini + descriptions"}.get(GEMINI_HELP, "")
PLAIN_COLUMNS = ["image", "model", "answer", "pred_label", "confidence", "clear_enough", "seconds", "time"]
HELP_COLUMNS = ["image", "model", "help", "answer", "pred_label", "confidence", "clear_enough",
                "references", "seconds", "time"]

class DailyLimit(Exception): pass
class KeyProblem(Exception): pass
class EmptyReply(Exception): pass
class NotInFreeTier(Exception): pass

def get_api_key():
    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        try:
            from google.colab import userdata
            k = userdata.get(name)
            if k:
                return k.strip()
        except Exception:
            pass
        if os.environ.get(name):
            return os.environ[name].strip()
    from getpass import getpass
    print("No Colab secret called GEMINI_API_KEY was found (see the steps at the top).")
    return getpass("Paste your Gemini API key here and press Enter (leave empty to skip Gemini): ").strip()

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

NORM_OPTIONS = {norm_text(o): o for o in OPTIONS}

def to_key(answer):
    if answer in OPTION_TO_KEY:
        return OPTION_TO_KEY[answer]
    n = norm_text(answer)
    if n in NORM_OPTIONS:
        return OPTION_TO_KEY[NORM_OPTIONS[n]]
    hits = [o for k, o in NORM_OPTIONS.items() if k and k in n]
    return OPTION_TO_KEY[max(hits, key=len)] if hits else ""

def parse_reply(text):
    text = (text or "").strip()
    data = {}
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            data = json.loads(m.group(0))
        except Exception:
            data = {}
    if not isinstance(data, dict):
        data = {}
    answer = data.get("diagnosis") or text
    try:
        conf = float(data.get("confidence"))
        conf = conf / 100 if 1 < conf <= 100 else conf
        conf = min(max(conf, 0.0), 1.0)
    except Exception:
        conf = ""
    clear = data.get("clear_enough", "")
    if isinstance(clear, str):
        clear = {"true": True, "yes": True, "false": False, "no": False}.get(clear.strip().lower(), "")
    return str(answer), to_key(answer), conf, clear

def append_row(path, row, columns):
    new = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns)
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in columns})

def load_answers(path):
    # the saved answers of GEMINI_MODEL in one file, one row per image
    if not os.path.exists(path):
        return pd.DataFrame(columns=["image", "model", "answer", "pred_label", "confidence", "clear_enough"])
    c = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not {"image", "model", "answer", "pred_label"} <= set(c.columns):
        print("Ignoring", os.path.basename(path), "(unexpected columns)")
        return pd.DataFrame(columns=["image", "model", "answer", "pred_label", "confidence", "clear_enough"])
    c = c[(c.model == GEMINI_MODEL) & ((c.pred_label != "") | (c.answer != ""))]
    return c.drop_duplicates("image", keep="last")

# ---- reuse the answers the loop notebook already got for plain Gemini (same model, same images,
#      same prompt, each level on its own), so nothing is asked twice ----
LOOP_CALLS = os.path.join(RESULTS_DIR, "loop_v2", "gemini_v2_calls.csv")

def import_loop_answers():
    if not os.path.exists(LOOP_CALLS):
        return 0
    L = pd.read_csv(LOOP_CALLS, dtype=str, keep_default_na=False)
    need = {"model", "kind", "view", "photo_id", "level", "answer", "pred_label", "confidence", "clear_enough", "time"}
    if need - set(L.columns):
        return 0
    L = L[(L.model == GEMINI_MODEL) & (L.kind == "alone") & (L.view == "original")]
    to_image = {(r.photo_id, r.level): r.image for r in meta.itertuples(index=False)}
    have = set(load_answers(GEMINI_CACHE).image)
    n = 0
    for r in L.itertuples(index=False):
        img = to_image.get((r.photo_id, r.level))
        if img and img not in have:
            append_row(GEMINI_CACHE, {"image": img, "model": GEMINI_MODEL, "answer": r.answer, "pred_label": r.pred_label,
                                      "confidence": r.confidence, "clear_enough": r.clear_enough, "seconds": "",
                                      "time": r.time}, PLAIN_COLUMNS)
            have.add(img)
            n += 1
    return n

# ---- which images to ask about: a fixed random order of photos, each with its levels ----
photos = meta.drop_duplicates("photo_id")[["photo_id", "disease"]]
rng = random.Random(SEED)
keep = []
for d, ids in photos.groupby("disease").photo_id:
    ids = sorted(ids)
    rng.shuffle(ids)
    keep += ids if GEMINI_PHOTOS_PER_DISEASE is None else ids[:GEMINI_PHOTOS_PER_DISEASE]
keep = sorted(keep)
random.Random(SEED + 1).shuffle(keep)
by_photo = {pid: g for pid, g in meta.groupby("photo_id")}
queue = [r for pid in keep for r in by_photo[pid].sort_values("level").itertuples(index=False)
         if r.level in GEMINI_LEVELS]
QUEUE_IMAGES = {r.image for r in queue}
PATH_OF = dict(zip(meta.image, meta.path))
print(f"{len(queue)} images in the Gemini test ({len(keep)} photos x {len(GEMINI_LEVELS)} levels).")

# ---- plain Gemini: reuse what is saved ----
if GEMINI_PLAIN != "off":
    n_reused = import_loop_answers()
    if n_reused:
        print(f"Reused {n_reused} plain Gemini answers from the loop notebook (same test), so they are not asked again.")
    n_plain = len(set(load_answers(GEMINI_CACHE).image) & QUEUE_IMAGES)
    print(f"Plain Gemini: {n_plain} of {len(queue)} images already answered (Step 2).")
    if n_plain == 0 and GEMINI_PLAIN == "reuse":
        print("  No saved plain Gemini answers were found, so the tables won't include plain Gemini. Upload "
              "gemini_3.5_flash_lite_predictions.csv from Step 2, or set GEMINI_PLAIN = 'run' (uses quota).")

# ---- the reference photos: exactly the clean photos the CLIP database voted with ----
REFS = {}
if GEMINI_HELP != "off" and HELP_REFS:
    if os.path.exists(NEIGHBOURS_FILE):
        nbr = pd.read_csv(NEIGHBOURS_FILE, keep_default_na=False)
        nbr = nbr[nbr.clip_model == CLIP_MODEL].sort_values(["image", "rank"])
        for img, g in nbr.groupby("image"):
            REFS[img] = list(zip(g.ref_image, g.ref_disease))[:HELP_REFS]
    n_missing = sum(1 for r in queue if len(REFS.get(r.image, [])) < HELP_REFS)
    if n_missing:
        print(f"{n_missing} images have no saved reference photos (did the CLIP section run?). "
              "Gemini with help skips them until CLIP has run.")
    if queue and queue[0].image in REFS:
        print("\nExample: reference photos for", queue[0].image)
        for r, (img, dis) in enumerate(REFS[queue[0].image], 1):
            print(f"  {r}. {img}  ->  {KEY_TO_OPTION.get(dis, dis)}")

# ---- the prompts ----
PLAIN_PROMPT = (   # exactly the Step 2 prompt
    "You are an expert plant pathologist. The photo shows part of a crop plant. "
    "It may be blurry, dark, grainy or compressed.\n"
    "Which ONE option below best describes the main disease, pest damage or condition you can see? "
    "Pick the single most likely option, even if you are unsure.\n\nOptions:\n"
    + "\n".join("- " + o for o in OPTIONS)
    + "\n\nAnswer in JSON with \"diagnosis\" (exactly one option from the list), \"confidence\" "
      "(from 0 to 1) and \"clear_enough\" (true if the photo is clear enough for a reliable diagnosis, "
      "false if the farmer should take it again)."
)

def option_lines(with_descriptions):
    if not with_descriptions:
        return "\n".join("- " + o for o in OPTIONS)
    return "\n".join(f"- {o} — {DESCRIPTIONS[k]}" if DESCRIPTIONS.get(k) else f"- {o}"
                     for o, k in zip(OPTIONS, LABEL_KEYS))

HELP_HEAD = ("You are an expert plant pathologist. The first image is a farmer's photo of part of a crop plant. "
             "It may be blurry, dark, grainy or compressed.\n")
if HELP_REFS:
    HELP_HEAD += (f"After it come {HELP_REFS} reference photos from a labelled collection of clean photos, each with "
                  "its correct label. A similarity search picked them as the photos that look most like the farmer's "
                  "photo, most similar first. Use them as examples to compare with: they can show a different "
                  "disease from the farmer's photo.\n")
HELP_QUESTION = (
    "Which ONE option below best describes the main disease, pest damage or condition you can see in the "
    "farmer's photo? Pick the single most likely option, even if you are unsure.\n\nOptions"
    + (", each with what it looks like" if HELP_DESC else "") + ":\n" + option_lines(HELP_DESC)
    + "\n\nAnswer in JSON with \"diagnosis\" (exactly one option from the list"
    + (", without its description" if HELP_DESC else "") + "), \"confidence\" (from 0 to 1) and \"clear_enough\" "
      "(true if the farmer's photo is clear enough for a reliable diagnosis, false if the farmer should take it again)."
)
if GEMINI_HELP != "off":
    print("\nWhat Gemini with help is told (the images go between these texts):\n")
    print(HELP_HEAD + "Farmer's photo: [image]\n"
          + "".join(f"Reference photo {r}{' (most similar)' if r == 1 else ''}. Label: ... [image]\n"
                    for r in range(1, HELP_REFS + 1))
          + HELP_QUESTION[:600] + " ...")

MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}
_ref_bytes = {}

def photo_part(path):
    with open(path, "rb") as f:
        data = f.read()
    return types.Part.from_bytes(data=data, mime_type=MIME.get(os.path.splitext(path)[1].lower(), "image/png"))

def reference_part(image):
    if image not in _ref_bytes:
        img = Image.open(PATH_OF[image]).convert("RGB")
        img.thumbnail((REFERENCE_MAX_SIDE, REFERENCE_MAX_SIDE))
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=90)
        _ref_bytes[image] = buf.getvalue()
    return types.Part.from_bytes(data=_ref_bytes[image], mime_type="image/jpeg")

def plain_contents(row):
    return [PLAIN_PROMPT, photo_part(row.path)]

def help_contents(row):
    parts = [HELP_HEAD, "Farmer's photo:", photo_part(row.path)]
    for r, (img, dis) in enumerate(REFS.get(row.image, [])[:HELP_REFS], 1):
        parts += [f"Reference photo {r}{' (most similar)' if r == 1 else ''}. Label: {KEY_TO_OPTION.get(dis, dis)}",
                  reference_part(img)]
    parts.append(HELP_QUESTION)
    return parts

def help_extra(row):
    refs = REFS.get(row.image, [])[:HELP_REFS]
    return {"help": GEMINI_HELP, "references": "; ".join(f"{img} = {dis}" for img, dis in refs)}

# ---- talking to Gemini: pacing and retries ----
_last_call = [0.0]
_rpm = [float(GEMINI_REQUESTS_PER_MINUTE)]   # slows down by itself if Google says "too many requests"
_busy_in_a_row, _ok_streak = [0], [0]

def pace():
    gap = 60.0 / max(_rpm[0], 0.1)
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

def ask(contents):
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=CONFIG)
    try:
        text = resp.text
    except Exception:
        text = None
    if not text:
        why = ""
        try:
            why = str(resp.candidates[0].finish_reason)
        except Exception:
            pass
        try:
            why += " " + str(resp.prompt_feedback.block_reason)
        except Exception:
            pass
        raise EmptyReply(why.strip() or "empty reply")
    return text

def retry_seconds(msg):
    m = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", msg) or re.search(r"retry in (\d+(?:\.\d+)?)\s*s", msg, re.I)
    return float(m.group(1)) if m else None

def call_gemini(contents, tries=8):
    last = None
    for attempt in range(tries):
        pace()
        try:
            text = ask(contents)
            _busy_in_a_row[0] = 0
            _ok_streak[0] += 1
            if _ok_streak[0] % 20 == 0:   # speed back up after a calm spell
                _rpm[0] = min(float(GEMINI_REQUESTS_PER_MINUTE), _rpm[0] * 1.15)
            return text
        except genai_errors.APIError as e:
            code, msg, last = getattr(e, "code", None), str(e), e
            if code == 429:
                if re.search(r"limit:\s*0\b", msg):
                    raise NotInFreeTier(msg)
                if re.search(r"PerDay|per day|daily", msg, re.I):
                    raise DailyLimit(msg)
                _busy_in_a_row[0] += 1
                _ok_streak[0] = 0
                if _busy_in_a_row[0] >= 8:   # never lets up: almost certainly the daily quota
                    raise DailyLimit(msg)
                if _rpm[0] > 2:
                    _rpm[0] = max(2.0, _rpm[0] * 0.8)
                wait = (retry_seconds(msg) or 20 * (attempt + 1)) + 2
                print(f"  too many requests; waiting {wait:.0f} s, then continuing at {_rpm[0]:.1f} per minute ...")
                time.sleep(wait)
                continue
            if code is None or code >= 500:
                wait = min(120, 5 * 2 ** attempt)
                print(f"  Gemini is busy ({code}); trying again in {wait} s ...")
                time.sleep(wait)
                continue
            if code in (401, 403) or re.search(r"API.?key", msg, re.I):
                raise KeyProblem(msg)
            raise   # e.g. 400 bad request or 404 model not found: a problem with this request
        except EmptyReply as e:
            last = e
            if attempt >= 1:
                raise
        except Exception as e:   # network hiccup
            last = e
            wait = min(120, 5 * 2 ** attempt)
            print(f"  connection problem ({short(e, 80)}); trying again in {wait} s ...")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {tries} tries: {short(last, 200)}")

def run_condition(name, cache_path, columns, build, extra=None):
    # asks Gemini about every image in the queue that has no saved answer in cache_path
    done = set(load_answers(cache_path).image)
    todo = [r for r in queue if r.image not in done]
    if build is help_contents and HELP_REFS:
        todo = [r for r in todo if len(REFS.get(r.image, [])) >= HELP_REFS]
    if not todo:
        print(f"{name}: every image is answered already.")
        return "done"
    print(f"\n{name}: {len(todo)} images to go, about {len(todo) / _rpm[0]:.0f} minutes at {_rpm[0]:.0f} per minute. "
          "You can stop at any time; progress is saved.")
    n_ok, n_err, errors_in_a_row, status = 0, 0, 0, "done"
    bar = tqdm(total=len(todo), desc=name)
    try:
        for row in todo:
            t0 = time.time()
            try:
                text = call_gemini(build(row))
                answer, key, conf, clear = parse_reply(text)
                rec = {"image": row.image, "model": GEMINI_MODEL, "answer": answer[:300], "pred_label": key,
                       "confidence": conf, "clear_enough": clear, "seconds": round(time.time() - t0, 1),
                       "time": datetime.datetime.now().isoformat(timespec="seconds")}
                if extra:
                    rec.update(extra(row))
                append_row(cache_path, rec, columns)
                n_ok += 1
                errors_in_a_row = 0
            except DailyLimit:
                print("\nGemini's free quota for today seems to be used up (it keeps answering 'too many requests'). "
                      "Progress is saved: run this notebook again after midnight Pacific time "
                      "(about 2 pm in Bangladesh) and it continues where it stopped.")
                status = "daily"
                break
            except KeyProblem as e:
                print("\nGemini rejected the API key:", short(e, 200))
                print("Check the GEMINI_API_KEY secret (steps at the top), then run this cell again.")
                status = "key"
                break
            except NotInFreeTier as e:
                print(f"\n{GEMINI_MODEL} isn't available on your free key:", short(e, 200))
                status = "key"
                break
            except Exception as e:
                n_err += 1
                errors_in_a_row += 1
                append_row(os.path.join(OUT_DIR, "gemini_errors.csv"),
                           {"image": row.image, "model": GEMINI_MODEL, "predictor": name, "error": short(e, 500),
                            "time": datetime.datetime.now().isoformat(timespec="seconds")},
                           ["image", "model", "predictor", "error", "time"])
                if n_ok == 0 and errors_in_a_row >= 2:
                    print("\nThe first requests failed, so Gemini stops for now:", short(e, 300))
                    status = "stopped"
                    break
                if errors_in_a_row >= 10:
                    print("\n10 errors in a row, so Gemini stops for now. Last error:", short(e, 200))
                    print("Run this cell again later; finished answers are kept.")
                    status = "stopped"
                    break
            bar.update(1)
    except KeyboardInterrupt:
        print("\nStopped. Answers so far are saved; run this cell again to continue.")
        status = "stopped"
    bar.close()
    total = len(set(load_answers(cache_path).image) & QUEUE_IMAGES)
    print(f"{name}: {n_ok} answers this run, {n_err} errors. {total} of {len(queue)} images answered in total.")
    return status

# ---- run: Gemini with help first (the new test), then plain Gemini if asked to ----
jobs = []
if GEMINI_HELP != "off":
    if HELP_REFS and not REFS:
        print("No reference photos are saved yet, so Gemini with help can't run. Run the CLIP section first "
              "(it needs to work once), or set GEMINI_HELP = 'descriptions'.")
    else:
        jobs.append((HELP_NAME, HELP_CACHE, HELP_COLUMNS, help_contents, help_extra))
if GEMINI_PLAIN == "run":
    jobs.append((PLAIN_NAME, GEMINI_CACHE, PLAIN_COLUMNS, plain_contents, None))

if RUN_GEMINI and jobs:
    API_KEY = get_api_key()
    if not API_KEY:
        print("No API key, so Gemini is skipped. The tables use the answers saved so far.")
    else:
        from google import genai
        from google.genai import types, errors as genai_errors
        client = genai.Client(api_key=API_KEY)
        CONFIG = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "diagnosis": types.Schema(type=types.Type.STRING, enum=OPTIONS),
                    "confidence": types.Schema(type=types.Type.NUMBER),
                    "clear_enough": types.Schema(type=types.Type.BOOLEAN),
                },
                required=["diagnosis", "confidence", "clear_enough"],
            ),
        )
        for job in jobs:
            if run_condition(*job) in ("daily", "key"):
                break
elif not RUN_GEMINI:
    print("RUN_GEMINI is off, so no new Gemini answers; the tables use the answers saved so far.")

## 5. Results for the weekly update

**Accuracy** is the share of images where the predictor named the right disease. All predictors are compared on the **same images**. If Gemini hasn't finished every image yet, the tables use only the images both Gemini predictors have done, and the image count is shown under each table.

How precise are the numbers? With about 200 images per level, a difference smaller than about **±7 points** could be chance. Within one group or one disease, which has fewer images, the margin is about ±10 to 20 points.

The notebook saves (in `SafeCrop/results/gemini_help`):

* `table1_accuracy_by_level.png`: main table, accuracy at each noise level
* `table2_info_rich_vs_scarce.png`: the information-richness comparison
* `table3_by_disease.png`, `table4_by_noise_type.png`
* `chart_accuracy_vs_noise.png`
* `table5_gemini_help_change.png`: **new**, how many points the reference photos and descriptions added to Gemini, overall and for info-rich vs info-scarce
* `table6_gemini_vs_database.png`: **new**, how often Gemini with help agreed with the CLIP database, and who was right when they disagreed
* `results_tables_gemini_help.xlsx`: every table plus extras (local versus foreign, crop accuracy, error types, Gemini's "retake?" answers, all predictions)

In [ ]:
# ---------- style (light chart surface, hairline grid, fixed colour per predictor) ----------
INK, INK2, MUTED, GRID, BASE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
LEVELS = ["L0", "L1", "L2", "L3"]
LEVEL_LABEL = {"L0": "Clean (L0)", "L1": "Mild (L1)", "L2": "Medium (L2)", "L3": "Severe (L3)"}
NOISE_NAME = {"gaussian_noise": "Sensor noise", "motion_blur": "Hand-shake blur", "defocus_blur": "Out of focus",
              "low_light": "Low light", "jpeg_compression": "Compression"}
def colour(m):
    return {"CLIP zero-shot": "#2a78d6", "CLIP photo database": "#eb6834",
            "Gemini 3.5 Flash-Lite": "#1baf7a"}.get(m, "#7c5cd6")
def short_name(m):
    return {"CLIP zero-shot": "CLIP zero-shot", "CLIP photo database": "CLIP database",
            "Gemini 3.5 Flash-Lite": "Gemini plain"}.get(m, "Gemini + help")
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.edgecolor": BASE, "axes.labelcolor": INK2,
                     "xtick.color": MUTED, "ytick.color": MUTED, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE})

# ---------- collect every prediction ----------
frames = []
if os.path.exists(CLIP_FILE):
    c = pd.read_csv(CLIP_FILE, keep_default_na=False)
    frames.append(c[["image", "model", "pred_label", "confidence"]])
gem_used = GEMINI_MODEL                               # only Gemini 3.5 Flash-Lite, never another model
gem_files = []
if GEMINI_PLAIN != "off":
    gem_files.append((PLAIN_NAME, GEMINI_CACHE))
if GEMINI_HELP != "off":
    gem_files.append((HELP_NAME, HELP_CACHE))
for name, path in gem_files:
    g = load_answers(path)
    g = g[g.image.isin(QUEUE_IMAGES)].copy()           # only the images in this notebook's Gemini test
    if len(g):
        g["model"] = name
        frames.append(g[["image", "model", "pred_label", "confidence", "clear_enough"]])
if not frames:
    raise RuntimeError("No predictions yet: run the CLIP and/or Gemini sections first.")

P = pd.concat(frames, ignore_index=True)
P = P.merge(meta[["image", "photo_id", "disease", "disease_name", "group", "origin", "noise", "level"]], on="image")
P["pred_label"] = P.pred_label.fillna("").astype(str)
P["confidence"] = pd.to_numeric(P.confidence, errors="coerce")
P["correct"] = P.pred_label == P.disease
P["true_crop"] = P.disease.map(LABEL_CROP)
P["pred_crop"] = P.pred_label.map(LABEL_CROP)
P["crop_correct"] = P.true_crop == P.pred_crop
P["outcome"] = np.select(
    [P.correct, P.pred_label == "", (P.pred_crop == P.true_crop) & P.pred_label.str.endswith("_healthy"),
     P.pred_crop == P.true_crop],
    ["Correct", "No usable answer", "Said healthy", "Wrong disease, right crop"], default="Wrong crop")

MODELS = [m for m in ["CLIP zero-shot", "CLIP photo database", PLAIN_NAME, HELP_NAME] if m and m in set(P.model)]
CHOICES = {m: (meta.disease.nunique() if m == "CLIP photo database" else len(LABELS)) for m in MODELS}

coverage = P.groupby("model").image.nunique().reindex(MODELS)
common = P.groupby("image").model.nunique()
common = set(common.index[common == len(MODELS)])
D = P[P.image.isin(common)].copy()
n_img = len(common)
print("Images answered:", ", ".join(f"{m}: {coverage[m]}" for m in MODELS))
if n_img < len(meta):
    print(f"-> The tables compare the {n_img} images that every predictor has answered (out of {len(meta)}).")
if n_img == 0:
    raise RuntimeError("No image has an answer from every predictor yet. Run the CLIP and Gemini sections first "
                       "(or check GEMINI_LEVELS), then run this cell again.")
if n_img < 400:
    print("   With this few images the numbers are rough; they will settle as Gemini finishes the rest.")

# ---------- tables ----------
def acc_table(df, rows):
    t = df.groupby(rows + ["level"])["correct"].mean().mul(100).unstack("level").reindex(columns=LEVELS)
    t["All levels"] = df.groupby(rows)["correct"].mean().mul(100)
    return t

t1 = acc_table(D, ["model"]).reindex(MODELS)
t1["Drop L0→L3 (points)"] = t1["L0"] - t1["L3"]
t1.insert(0, "Choices", [CHOICES[m] for m in t1.index])
t1 = t1.rename(columns=LEVEL_LABEL)

t2 = acc_table(D, ["model", "group"]).reindex(MODELS, level=0).rename(columns=LEVEL_LABEL)
t5 = acc_table(D, ["model", "origin"]).reindex(MODELS, level=0).rename(columns=LEVEL_LABEL)

D["state"] = np.where(D.level == "L0", "clean", "noisy")
t3 = D.groupby(["group", "disease_name", "origin", "model", "state"])["correct"].mean().mul(100).unstack(["model", "state"])
cols3 = [(m, s) for m in MODELS for s in ["clean", "noisy"] if (m, s) in t3.columns]
t3 = t3[cols3]
t3.columns = [f"{short_name(m)}\n{s}" for m, s in cols3]

noisy = D[D.level != "L0"]
t4 = noisy.groupby(["noise", "model"])["correct"].mean().mul(100).unstack("model").reindex(columns=MODELS)
t4.index = [NOISE_NAME.get(n, n) for n in t4.index]

t6 = D.groupby(["model", "level"])["crop_correct"].mean().mul(100).unstack("level").reindex(MODELS).rename(columns=LEVEL_LABEL)
t7 = (D.groupby(["model", "outcome"]).size() / D.groupby("model").size()).mul(100).unstack("outcome").reindex(MODELS).fillna(0)
t7 = t7[[c for c in ["Correct", "Wrong disease, right crop", "Said healthy", "Wrong crop", "No usable answer"] if c in t7.columns]]
gem_rows = D[D.model.str.startswith("Gemini")]
t8 = None
if len(gem_rows) and "clear_enough" in gem_rows:
    ce = gem_rows.clear_enough.astype(str).str.lower().map({"true": 1.0, "false": 0.0})
    t8 = pd.DataFrame({
        "Gemini said 'clear enough' (%)": ce.groupby(gem_rows.level).mean().mul(100),
        "Accuracy when 'clear enough' (%)": gem_rows[ce == 1].groupby("level").correct.mean().mul(100),
        "Accuracy when 'retake' (%)": gem_rows[ce == 0].groupby("level").correct.mean().mul(100),
    }).reindex(LEVELS).rename(index=LEVEL_LABEL)

# ---------- new: what the help changed for Gemini, and did it just copy the CLIP database? ----------
t9 = t10 = None
if PLAIN_NAME in MODELS and HELP_NAME in MODELS:
    rows = {"All diseases": acc_table(D, ["model"])}
    for grp_name in sorted(D.group.astype(str).unique()):
        rows[grp_name] = acc_table(D[D.group.astype(str) == grp_name], ["model"])
    t9 = pd.DataFrame({k: v.loc[HELP_NAME] - v.loc[PLAIN_NAME] for k, v in rows.items()}).T
    t9 = t9.rename(columns=LEVEL_LABEL)
    t9.index.name = "Change in accuracy (points)"
if HELP_NAME in MODELS and "CLIP photo database" in MODELS:
    h = D[D.model == HELP_NAME][["image", "level", "pred_label", "correct"]]
    b = D[D.model == "CLIP photo database"][["image", "pred_label", "correct"]].rename(
        columns={"pred_label": "db_pred", "correct": "db_correct"})
    hb = h.merge(b, on="image")
    hb["agree"] = hb.pred_label == hb.db_pred
    t10 = pd.DataFrame({
        "Gemini agreed with the database (%)": hb.groupby("level").agree.mean().mul(100),
        "Gemini right when it agreed (%)": hb[hb.agree].groupby("level").correct.mean().mul(100),
        "Gemini right when it disagreed (%)": hb[~hb.agree].groupby("level").correct.mean().mul(100),
        "Database right when they disagreed (%)": hb[~hb.agree].groupby("level").db_correct.mean().mul(100),
    }).reindex(LEVELS).rename(index=LEVEL_LABEL)

for title, t in [("1. Accuracy (%) by noise level", t1), ("2. Info-rich (A) vs info-scarce (B), accuracy (%)", t2),
                 ("3. Accuracy (%) by disease, clean (L0) vs noisy (L1-L3)", t3),
                 ("4. Accuracy (%) on noisy images (L1-L3) by noise type", t4)]:
    print("\n" + title)
    display(t.round(0))

# ---------- a few plain-language lines for the update ----------
lines = []
def level_text(row):
    # "67% on clean photos, 13% at severe noise (drop of 54 points)", using only the levels that have answers
    c, v = row.get("Clean (L0)"), row.get("Severe (L3)")
    if pd.notna(c) and pd.notna(v):
        return f"{c:.0f}% on clean photos, {v:.0f}% at severe noise (drop of {c - v:.0f} points)"
    parts = [f"{row[LEVEL_LABEL[l]]:.0f}% at {LEVEL_LABEL[l].split(' ')[0].lower()}" for l in LEVELS
             if pd.notna(row.get(LEVEL_LABEL[l]))]
    return ", ".join(parts) if parts else "no answers"
def ends(row):
    vals = [row[LEVEL_LABEL[l]] for l in LEVELS if pd.notna(row.get(LEVEL_LABEL[l]))]
    return f"{vals[0]:.0f}% → {vals[-1]:.0f}%" if len(vals) > 1 else (f"{vals[0]:.0f}%" if vals else "–")
for m in MODELS:
    s = f"{m}: {level_text(t1.loc[m])}"
    a = t2.loc[m] if m in t2.index.get_level_values(0) else None
    if a is not None:
        ga = [i for i in a.index if str(i).startswith("A")]
        gb = [i for i in a.index if str(i).startswith("B")]
        if ga and gb:
            s += f"; info-rich {ends(a.loc[ga[0]])}, info-scarce {ends(a.loc[gb[0]])}"
    lines.append(s + ".")
if t9 is not None:
    display_title = "5. What the help changed: Gemini with help minus plain Gemini (points)"
    print("\n" + display_title)
    display(t9.round(0))
    lines.append(f"{HELP_NAME} vs plain Gemini: {t9.loc['All diseases', 'All levels']:+.0f} points overall"
                 + "".join(f", {LEVEL_LABEL[l].split(' ')[0].lower()} {t9.loc['All diseases', LEVEL_LABEL[l]]:+.0f}"
                           for l in LEVELS if pd.notna(t9.loc['All diseases', LEVEL_LABEL[l]])) + ".")
if t10 is not None:
    print("\n6. Did Gemini with help just copy the CLIP database?")
    display(t10.round(0))
print("\nSummary")
for l in lines:
    print(" •", l)

In [ ]:
# ---------- tables as pictures for the slides ----------
def tint(v):
    # 0% = chart surface, 100% = light blue; text stays dark so it is always readable
    if v is None or pd.isna(v):
        return SURFACE
    t = min(max(float(v) / 100.0, 0.0), 1.0)
    a, b = np.array([252, 252, 251]), np.array([0x86, 0xb6, 0xef])
    r, g, bl = (a + (b - a) * t).round().astype(int)
    return f"#{r:02x}{g:02x}{bl:02x}"

def render_table(df, path, title, note="", pct=(), pts=(), center=()):
    df = df.reset_index()
    df.columns = [str(c) for c in df.columns]
    pct, pts = set(pct), set(pts)
    def fmt(c, v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "–"
        if c in pct:
            return f"{v:.0f}%"
        if c in pts:
            return f"{v:+.0f}" if round(v) != 0 else "0"
        return str(v)
    text = [[fmt(c, v) for c, v in zip(df.columns, row)] for row in df.itertuples(index=False)]
    heads = list(df.columns)
    widths = []
    for j, h in enumerate(heads):
        longest = max([len(x) for x in h.split("\n")] + [len(r[j]) for r in text] + [3])
        widths.append(max(0.75, longest * 0.085 + 0.35))
    row_h, head_h = 0.34, (0.55 if any("\n" in h for h in heads) else 0.38)
    top, bottom, pad = 0.85, (0.55 if note else 0.2), 0.25
    W, H = sum(widths) + 2 * pad, top + head_h + row_h * len(text) + bottom
    fig = plt.figure(figsize=(W, H), dpi=200)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis("off")
    ax.text(pad, 0.36, title, fontsize=13, fontweight="bold", color=INK, va="center")
    ax.text(pad, 0.66, f"{n_img} images · {datetime.date.today():%d %b %Y}", fontsize=8.5, color=INK2, va="center")
    xs = np.cumsum([pad] + widths[:-1])
    numeric = [c in pct or c in pts or c in center for c in heads]
    for j, h in enumerate(heads):
        x = xs[j] + (0.08 if not numeric[j] else widths[j] / 2)
        ax.text(x, top + head_h / 2, h, fontsize=9, color=INK2, fontweight="bold",
                ha="left" if not numeric[j] else "center", va="center", linespacing=1.15)
    ax.plot([pad, W - pad], [top + head_h, top + head_h], color=BASE, lw=1)
    for i, r in enumerate(text):
        y = top + head_h + i * row_h
        for j, val in enumerate(r):
            c = heads[j]
            if c in pct:
                ax.add_patch(matplotlib.patches.FancyBboxPatch(
                    (xs[j] + 0.06, y + 0.045), widths[j] - 0.12, row_h - 0.09,
                    boxstyle="round,pad=0,rounding_size=0.04", fc=tint(df.iloc[i][c]), ec="none"))
            ax.text(xs[j] + (0.08 if not numeric[j] else widths[j] / 2), y + row_h / 2, val, fontsize=9.5,
                    color=INK, ha="left" if not numeric[j] else "center", va="center")
        ax.plot([pad, W - pad], [y + row_h, y + row_h], color=GRID, lw=0.7)
    if note:
        ax.text(pad, H - bottom / 2, note, fontsize=8, color=MUTED, va="center")
    fig.savefig(path, dpi=200, facecolor=SURFACE)
    plt.close(fig)

FIG = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG, exist_ok=True)
lv = list(LEVEL_LABEL.values())

d1 = t1.copy(); d1.index.name = "Predictor"
render_table(d1, os.path.join(FIG, "table1_accuracy_by_level.png"), "SafeCrop: accuracy by noise level",
             note="Accuracy = % of images where the predictor named the right disease. Differences under ~7 points may be chance.",
             pct=lv + ["All levels"], pts=["Drop L0→L3 (points)"], center=["Choices"])

d2 = t2.copy(); d2.index.names = ["Predictor", "Group"]
render_table(d2, os.path.join(FIG, "table2_info_rich_vs_scarce.png"), "Info-rich (A) vs info-scarce (B): accuracy by noise level",
             note="Group A: apple scab, potato late blight, rice stem borer. Group B: grape leaf blight, pear leaf spot, jute mosaic, jute dieback.",
             pct=lv + ["All levels"])

d3 = t3.copy(); d3.index.names = ["Group", "Disease", "Origin"]
render_table(d3, os.path.join(FIG, "table3_by_disease.png"), "Accuracy by disease: clean (L0) vs noisy (L1-L3)",
             note="Few photos per disease (25-35), so single-disease numbers can move ±15 points by chance.",
             pct=list(t3.columns))

d4 = t4.copy(); d4.index.name = "Noise type"
d4.columns = [short_name(m) for m in d4.columns]
render_table(d4, os.path.join(FIG, "table4_by_noise_type.png"), "Accuracy on noisy images (L1-L3) by noise type",
             pct=list(d4.columns))

if t9 is not None:
    d9 = t9.copy(); d9.index.name = "Change in accuracy (points)"
    render_table(d9, os.path.join(FIG, "table5_gemini_help_change.png"),
                 f"What the help changed: {HELP_NAME} minus plain Gemini",
                 note="Positive = the help made Gemini more accurate. Under ~7 points (all diseases) may be chance.",
                 pts=list(d9.columns))
if t10 is not None:
    d10 = t10.copy(); d10.index.name = "Level"
    d10.columns = ["Agreed with\ndatabase", "Right when\nagreed", "Right when\ndisagreed", "Database right\nwhen disagreed"]
    render_table(d10, os.path.join(FIG, "table6_gemini_vs_database.png"),
                 "Did Gemini with help just copy the CLIP database?",
                 note="If Gemini only copied the reference photos' labels, it would agree almost always.",
                 pct=list(d10.columns))

# ---------- chart: accuracy vs noise level, all / info-rich / info-scarce ----------
panels = [("All diseases", D)]
for prefix, name in [("A", "Info-rich (group A)"), ("B", "Info-scarce (group B)")]:
    sub = D[D.group.astype(str).str.startswith(prefix)]
    if len(sub):
        panels.append((name, sub))
fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.3), sharey=True, dpi=200)
axes = np.atleast_1d(axes)
x = np.arange(4)
for k, (ax, (name, sub)) in enumerate(zip(axes, panels)):
    ends = []
    for m in MODELS:
        y = sub[sub.model == m].groupby("level").correct.mean().mul(100).reindex(LEVELS).to_numpy()
        ax.plot(x, y, color=colour(m), lw=2, marker="o", ms=8, mec=SURFACE, mew=2,
                solid_capstyle="round", solid_joinstyle="round", label=m, zorder=3)
        if not np.isnan(y[-1]):
            ends.append([max(y[-1], 3), short_name(m)])
    if k == 0:   # direct labels on the first panel, nudged apart so they don't collide
        ends.sort(key=lambda e: e[0])
        for i in range(1, len(ends)):
            if ends[i][0] - ends[i - 1][0] < 7:
                ends[i][0] = ends[i - 1][0] + 7
        for yv, lab in ends:
            ax.text(3.15, yv, lab, fontsize=8.5, color=INK2, va="center")
    ax.set_title(name, fontsize=11, color=INK, loc="left")
    ax.set_xticks(x, ["Clean\nL0", "Mild\nL1", "Medium\nL2", "Severe\nL3"])
    ax.set_xlim(-0.25, 3.9 if k == 0 else 3.25)
    ax.set_ylim(0, 100)
    ax.set_yticks(range(0, 101, 20))
    ax.grid(axis="y", color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0)
axes[0].set_ylabel("Accuracy (%)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=len(MODELS), frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(os.path.join(FIG, "chart_accuracy_vs_noise.png"), facecolor=SURFACE, bbox_inches="tight")
plt.close(fig)

for f in ["table1_accuracy_by_level.png", "table2_info_rich_vs_scarce.png", "chart_accuracy_vs_noise.png",
          "table5_gemini_help_change.png", "table6_gemini_vs_database.png",
          "table3_by_disease.png", "table4_by_noise_type.png"]:
    if os.path.exists(os.path.join(FIG, f)):
        display(ShowImage(filename=os.path.join(FIG, f), width=900))

# ---------- everything in one Excel file ----------
xlsx = os.path.join(OUT_DIR, "results_tables_gemini_help.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    pd.DataFrame({"summary": lines + [f"Images compared: {n_img}",
                                      f"CLIP model: {CLIP_MODEL}", f"Gemini model: {gem_used}",
                                      f"Gemini help: {GEMINI_HELP} ({HELP_REFS} reference photos)",
                                      f"Gemini levels: {', '.join(GEMINI_LEVELS)}"]}).to_excel(w, sheet_name="Summary", index=False)
    t1.round(1).to_excel(w, sheet_name="1 by level")
    t2.round(1).to_excel(w, sheet_name="2 info-rich vs scarce")
    t3.round(1).to_excel(w, sheet_name="3 by disease")
    t4.round(1).to_excel(w, sheet_name="4 by noise type")
    t5.round(1).to_excel(w, sheet_name="5 local vs foreign")
    t6.round(1).to_excel(w, sheet_name="6 crop right (%)")
    t7.round(1).to_excel(w, sheet_name="7 error types (%)")
    if t8 is not None:
        t8.round(1).to_excel(w, sheet_name="8 Gemini retake")
    if t9 is not None:
        t9.round(1).to_excel(w, sheet_name="9 help change (points)")
    if t10 is not None:
        t10.round(1).to_excel(w, sheet_name="10 Gemini vs database")
    P.drop(columns=["true_crop"]).to_excel(w, sheet_name="all predictions", index=False)
P.to_csv(os.path.join(OUT_DIR, "all_predictions.csv"), index=False)
print("Saved tables and figures in", OUT_DIR)

## 6. Download the results

This makes `SafeCrop_gemini_help_results.zip` with all tables, figures and predictions, and downloads it. If Drive is connected, everything is also in `My Drive/SafeCrop/results/gemini_help`.

In [ ]:
zip_base = os.path.join(WORK, "SafeCrop_gemini_help_results")
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Zip ready:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if DRIVE_OK:
    print("Also in Google Drive:", OUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download didn't start (" + short(e, 80) + "). Use the folder icon on the left to download it.")

## Notes

* **Same information, fair test:** Gemini with help gets exactly the clean photos the CLIP database votes with. If it does better than the database, it is using those examples better; if it agrees with the database almost every time, it is mostly copying the labels (table 6 shows which).
* **The references only show our 7 diseases.** That hints which answers are likely, for Gemini and for the CLIP database alike. Say so when you report the numbers.
* **To see what each kind of help does on its own,** run again with `GEMINI_HELP = "references"` and then `"descriptions"`. Each setting keeps its own answers file, so nothing is mixed. Each full run costs 840 requests (210 with `GEMINI_LEVELS = ["L3"]`).
* **Pear leaf spot:** these photos are 3968 × 2976 and become small for every model, so the small spots are hard to see even when clean.
* **Data sources differ per disease** (for example PlantVillage lab photos for potato and grape), so the CLIP database, and now Gemini with its reference photos, can partly match the photo *style* rather than the disease.